# CS-317: Reinforcement Learning — Lab 9
## Learning to Act: SARSA and Q-learning
**BS AI-1**

This notebook provides the loading-dock environment and helper functions. Complete every `# TODO` cell, run all cells top to bottom without errors, and answer the written questions in the markdown cells provided for them.

- Set your personal **SEED** (the last three digits of your roll number) in the first code cell below. Every random experiment must use it, and your written answers must refer to your own numbers.
- **Predict first** cells must be filled in *before* you run the code beneath them.
- Your code must learn only through `env.reset()` and `env.step()`; the rules of the dock are hidden.

Reinforcement-learning libraries may not be used. NumPy and Matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=2, suppress=True)

In [ ]:
# ---------------------------------------------------------------
# Personal seed: the last three digits of your roll number.
# Every random experiment in this notebook must use it.
# ---------------------------------------------------------------
SEED = None          # TODO: e.g. SEED = 417

assert SEED is not None, "Set SEED to the last three digits of your roll number before running the notebook."
print("SEED =", SEED)

---
## Provided Starter Code

Run the cells in this section as-is. You do not need to modify them.

| Name | What it gives you |
|---|---|
| `DockEnv()` | `reset()` → `s` ; `step(a)` → `s2, r, done`. The rules are hidden. |
| `START`, `GOAL`, `EDGE` | cell 24, cell 31, and the dock-edge cells 25–30 |
| `test_route(env, Q)` | follows the best action of `Q` from S with **no exploration**; returns `reward`, `steps`, `path`, `reached`, `top_row` |
| `plot_route(Q, title)` | draws the grid, the best action in every cell, and the route from S |
| `moving_average(x, w=10)` | smooths a learning curve |

A Q table is an array of shape `(32, 4)`: one row per cell, one column per action (Up, Down, Left, Right).

In [ ]:
# ---------------------------------------------------------------
# Warehouse loading dock (4 x 8 grid)
#
#        col: 0  1  2  3  4  5  6  7
#   row 0     .  .  .  .  .  .  .  .        cells are numbered row by row:
#   row 1     .  .  .  .  .  .  .  .        cell = 8 * row + col
#   row 2     .  .  .  .  .  .  .  .
#   row 3     S  X  X  X  X  X  X  G        S = start (cell 24), G = goal (cell 31)
#                                           X = dock edge (cells 25..30)
#
#   Actions: 0 = Up, 1 = Down, 2 = Left, 3 = Right
#   Every move costs -1. A move into the outer wall leaves the robot in place.
#   Driving onto the dock edge costs -100 and sends the robot back to S;
#   the episode continues. Reaching G ends the episode. No discounting (gamma = 1).
# ---------------------------------------------------------------

N_ROWS, N_COLS = 4, 8
N_STATES, N_ACTIONS = N_ROWS * N_COLS, 4
ACTIONS = ["Up", "Down", "Left", "Right"]
START, GOAL = 24, 31
EDGE = tuple(range(25, 31))
GAMMA = 1.0


class DockEnv:
    """The loading dock as an environment you can only ACT in.

    reset()       -> s                 the start cell (24)
    step(action)  -> s2, r, done       next cell, reward, end-of-episode flag

    The rules are hidden: reading env.P or env.R raises an error, as in Labs 7 and 8.
    """

    def __init__(self):
        self.__s = None

    def __getattr__(self, name):
        if name in ("P", "R", "transition_matrix"):
            raise AttributeError(f"The model is hidden: '{name}' is not available. "
                                 "This environment only offers reset() and step(action).")
        raise AttributeError(name)

    def reset(self):
        self.__s = START
        return self.__s

    def step(self, action):
        if self.__s is None or self.__s == GOAL:
            raise RuntimeError("Episode has ended (or not started): call reset() first.")
        r, c = divmod(self.__s, N_COLS)
        if action == 0:   r = max(r - 1, 0)
        elif action == 1: r = min(r + 1, N_ROWS - 1)
        elif action == 2: c = max(c - 1, 0)
        else:             c = min(c + 1, N_COLS - 1)
        s2 = r * N_COLS + c
        if s2 in EDGE:                       # drove onto the dock edge
            self.__s = START
            return START, -100.0, False
        self.__s = s2
        return s2, -1.0, s2 == GOAL


env = DockEnv()
print("Start cell:", env.reset())

In [ ]:
# ---------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------

def test_route(env, Q, max_steps=50):
    """Follows the greedy action argmax_a Q(s, a) from the start, with NO exploration.

    Returns a dict with
      'reward'  : total reward of the trip
      'steps'   : number of steps taken
      'path'    : list of visited cells, starting with the start cell
      'reached' : True if the goal was reached within max_steps
      'top_row' : the highest row the route climbs to (0 = top row); None if the goal was not reached
    """
    s = env.reset()
    path, total = [s], 0.0
    for _ in range(max_steps):
        a = int(np.argmax(Q[s]))
        s, r, done = env.step(a)
        total += r
        path.append(s)
        if done:
            rows = [p // N_COLS for p in path[1:-1]] or [N_ROWS - 1]
            return {"reward": total, "steps": len(path) - 1, "path": path,
                    "reached": True, "top_row": min(rows)}
    return {"reward": total, "steps": max_steps, "path": path, "reached": False, "top_row": None}


def plot_route(Q=None, title="", env=None):
    """Draws the grid with the greedy action of every cell (small grey arrows) and the
    route followed from S with no exploration (red line). With Q=None, only the layout is drawn."""
    fig, ax = plt.subplots(figsize=(8, 4.2))
    for s in range(N_STATES):
        r, c = divmod(s, N_COLS)
        face = "#f4b6b6" if s in EDGE else ("#c6e0b4" if s == GOAL else ("#bdd7ee" if s == START else "white"))
        ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, facecolor=face, edgecolor="grey"))
        if s in EDGE:
            ax.text(c, r, "X", ha="center", va="center", fontsize=12, color="#a33")
            continue
        if s == GOAL:
            ax.text(c, r, "G", ha="center", va="center", fontsize=13, fontweight="bold")
            continue
        if s == START:
            ax.text(c - .35, r - .3, "S", fontsize=10, fontweight="bold")
        if Q is None:
            continue
        a = int(np.argmax(Q[s]))
        dx, dy = [(0, -.3), (0, .3), (-.3, 0), (.3, 0)][a]
        ax.annotate("", xy=(c + dx, r + dy), xytext=(c, r),
                    arrowprops=dict(arrowstyle="->", color="grey", lw=1.2))
    status = None
    if Q is not None:
        route = test_route(env if env is not None else DockEnv(), Q)
        xs = [p % N_COLS for p in route["path"]]
        ys = [p // N_COLS for p in route["path"]]
        ax.plot(xs, ys, color="crimson", lw=3, alpha=.8)
        status = f"{route['steps']} steps, reward {route['reward']:.0f}" if route["reached"] else "did not reach G"
    ax.set_xlim(-.5, N_COLS - .5); ax.set_ylim(N_ROWS - .5, -.5)
    ax.set_xticks(range(N_COLS)); ax.set_yticks(range(N_ROWS))
    ax.set_xlabel("column"); ax.set_ylabel("row")
    ax.set_title(title if status is None else f"{title}  (route: {status})")
    ax.set_aspect("equal"); plt.tight_layout(); plt.show()


def moving_average(x, w=10):
    """Smooths a learning curve: the average of each window of w consecutive values."""
    x = np.asarray(x, dtype=float)
    return np.convolve(x, np.ones(w) / w, mode="valid")


# Show the empty grid
plot_route(title="Loading dock: S = start, G = goal, X = dock edge")

---
## Part 1: From Prediction to Control

Labs 7 and 8 **measured** how good a fixed policy was. In this lab the robot **improves its own behaviour**. To choose an action without knowing the rules, it learns an **action-value table** $Q(s, a)$: one number for every cell and every action.

**ε-greedy action choice.** With probability $1-\varepsilon$ the robot takes the action with the highest $Q(s,a)$ (ties broken at random); with probability $\varepsilon$ it takes a completely random action (any of the four, possibly the best one again).

### Task 1: Why Q, and ε-greedy

**(a)** In Lab 5, choosing the best action from $V$ required the rules. Explain why, and say what the robot must learn instead when the rules are hidden.

*your answer here*

**(b)** In some cell, $Q(s,\cdot)$ = (Up –3, Down –10, Left –6, Right –2). Complete Table 5 by hand.

| Action | Probability with ε = 0.1 | Probability with ε = 0.3 |
|---|---|---|
| Up | | |
| Down | | |
| Left | | |
| Right | | |

*your working here*

**(c)** Implement `epsilon_greedy(Q, s, epsilon, rng)`. Check it by choosing an action 10,000 times for the Q-values of part (b) with ε = 0.1 and comparing the frequencies with Table 5.

In [ ]:
def epsilon_greedy(Q, s, epsilon, rng):
    # With probability epsilon: a random action (0..3).
    # Otherwise: an action with the highest Q[s] (break ties at random).
    # TODO
    # your code here

    return a


Q_test = np.zeros((N_STATES, N_ACTIONS))
Q_test[0] = [-3, -10, -6, -2]
rng = np.random.default_rng(SEED)

# TODO: choose an action 10,000 times for cell 0 with epsilon = 0.1 and print the
#       frequency of each action.
# your code here

**Questions**

1. Why must the robot sometimes take a random action, even when it already knows a good one?

   *your answer here*

2. With ε = 0.1, how often does the robot take an action other than the best one? (Careful: the random choice can pick the best action too.)

   *your answer here*

### Task 2: SARSA and Q-learning by Hand

After every step $S_t, A_t \rightarrow R_{t+1}, S_{t+1}$, both methods choose the next action $A_{t+1}$ with ε-greedy and update

$$Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha\,[\,\text{target} - Q(S_t,A_t)\,]$$

| Method | Target |
|---|---|
| SARSA | $R_{t+1} + \gamma\, Q(S_{t+1}, A_{t+1})$ — the action it **will actually take** next |
| Q-learning | $R_{t+1} + \gamma\, \max_a Q(S_{t+1}, a)$ — the **best** action in the next cell |

If $S_{t+1}$ is the goal, the target is just $R_{t+1}$.

The robot is in cell 10, takes **Right**, receives $-1$ and lands in cell 11. Before the update, $Q(10, \text{Right}) = -5$, and in cell 11: $Q(11,\cdot)$ = (Up –7, Down –12, Left –9, Right –4). Use $\alpha = 0.5$ and $\gamma = 1$.

**Predict first.** In which of the two cases below will SARSA and Q-learning give different answers? Why?

*your prediction here*

**(a)** Table 6: complete both cases by hand.

| Case | Next action $A_{t+1}$ | SARSA target | SARSA new $Q(10,\text{Right})$ | Q-learning target | Q-learning new $Q(10,\text{Right})$ |
|---|---|---|---|---|---|
| A (greedy) | Right | | | | |
| B (exploring) | Down | | | | |

*your working here*

**(b)** Implement `sarsa_update` and `q_learning_update` and check both cases of Table 6.

In [ ]:
def sarsa_update(Q, s, a, r, s2, a2, done, alpha, gamma):
    # TODO: one SARSA update of Q[s, a]; if done, the target is just r.
    # your code here

    pass


def q_learning_update(Q, s, a, r, s2, done, alpha, gamma):
    # TODO: one Q-learning update of Q[s, a]; if done, the target is just r.
    # your code here

    pass


# TODO: set up Q[10, 3] = -5 and Q[11] = [-7, -12, -9, -4] in a fresh table for each
#       case and check Table 6 (alpha = 0.5, gamma = 1).
# your code here

**Questions**

1. In which case did the two methods agree, and why?

   *your answer here*

2. In case B, SARSA lowered $Q(10, \text{Right})$ but Q-learning did not. In words, what is each method assuming about how the robot will behave after reaching cell 11?

   *your answer here*

---
## Part 2: Learning on the Loading Dock

### Task 3: SARSA

Implement `sarsa(env, n_episodes, alpha, epsilon, gamma, rng)`. Start from $Q = 0$. In every episode, choose the first action with ε-greedy; after every step, choose the next action with ε-greedy, update with `sarsa_update`, and continue with the action you chose. Return the Q table, the total reward of every episode, and the number of falls (rewards of –100) in every episode.

In [ ]:
def sarsa(env, n_episodes, alpha, epsilon, gamma, rng):
    Q = np.zeros((N_STATES, N_ACTIONS))
    episode_rewards = np.zeros(n_episodes)
    episode_falls = np.zeros(n_episodes)
    for ep in range(n_episodes):
        s = env.reset()
        a = epsilon_greedy(Q, s, epsilon, rng)
        done = False
        while not done:
            s2, r, done = env.step(a)
            # TODO: record the reward (and a fall if r == -100), choose a2 with
            #       epsilon_greedy, update with sarsa_update, then move on: s, a = s2, a2
            # your code here

    return Q, episode_rewards, episode_falls

**Predict first.** Will the route SARSA learns run right along the dock edge, or further away from it? Why?

*your prediction here*

**(a)** Run SARSA for 500 episodes with α = 0.1, ε = 0.1, γ = 1 (`rng = np.random.default_rng(SEED)`). Plot the reward of every episode and its moving average.

In [ ]:
# TODO: run SARSA and plot the learning curve (reward per episode and moving average).
# your code here

**(b)** Draw the learned route with `plot_route`, and record `test_route` for it.

In [ ]:
# TODO: plot_route and test_route for the SARSA Q table.
# your code here

**Questions**

1. Roughly how many episodes did SARSA need before its reward stopped improving quickly? What were the early episodes like, and why?

   *your answer here*

2. Describe the route SARSA learned (number of steps, highest row). Is it the shortest possible route? If not, why would SARSA choose a longer one?

   *your answer here*

### Task 4: Q-learning, and the Comparison

Implement `q_learning(env, n_episodes, alpha, epsilon, gamma, rng)`. It is the same as `sarsa`, except that the update uses `q_learning_update`.

In [ ]:
def q_learning(env, n_episodes, alpha, epsilon, gamma, rng):
    # TODO: the same as sarsa, but update with q_learning_update.
    # your code here

    return Q, episode_rewards, episode_falls

**Predict first.** Which method will learn the shorter route? Which will collect more reward *while* it is learning?

*your prediction here*

**(a)** Run Q-learning with the same settings as Task 3 (seed SEED). Draw its route with `plot_route` and record `test_route`.

In [ ]:
# TODO: run Q-learning, plot_route and test_route.
# your code here

**(b)** Run each method 20 times (run $k$ uses `np.random.default_rng(SEED + k)`), 500 episodes each, with α = 0.1, ε = 0.1. Plot the moving average of the reward per episode, averaged over the 20 runs, one line per method. Complete Table 7, where "last 100 episodes" means episodes 401–500, and the last two rows come from `test_route` (exploration switched off).

In [ ]:
# TODO: 20 runs of each method, the averaged learning curves on one labelled plot,
#       and the numbers for Table 7.
# your code here

| Measure | SARSA | Q-learning |
|---|---|---|
| Average reward per episode, last 100 episodes (while exploring) | | |
| Average falls per episode, last 100 episodes | | |
| Most common route: steps and highest row | | |
| Average reward of the learned route with exploration off | | |

**Questions**

1. Which method learned the shorter route? Which collected more reward while learning? Quote Table 7.

   *your answer here*

2. Q-learning's route is shorter, yet it earned less reward while learning. Use the number of falls to explain why.

   *your answer here*

3. SARSA is called **on-policy** and Q-learning **off-policy**. Using your answer to Task 2, explain what these words mean.

   *your answer here*

---
## Part 3: Exploration

### Task 5: How Much Exploration?

**Predict first.** As ε grows, what will happen to SARSA's route? And to Q-learning's?

*your prediction here*

**(a)** For ε ∈ {0.05, 0.1, 0.2}, run each method 10 times (seeds SEED, …, SEED + 9; 500 episodes; α = 0.1). Record the average reward per episode over the last 100 episodes, and use `test_route` to count how many of the 10 learned routes reach their highest row in row 0, row 1 or row 2, or do not reach G.

In [ ]:
# TODO: Table 8 — for each epsilon and each method: average reward (last 100 episodes,
#       averaged over 10 runs) and the counts of routes by highest row.
# your code here

| ε | Method | Average reward (last 100 episodes) | Routes via row 0 | via row 1 | via row 2 | Did not reach G |
|---|---|---|---|---|---|---|
| 0.05 | SARSA | | | | | |
| 0.05 | Q-learning | | | | | |
| 0.1 | SARSA | | | | | |
| 0.1 | Q-learning | | | | | |
| 0.2 | SARSA | | | | | |
| 0.2 | Q-learning | | | | | |

**(b)** Plot the average reward per episode (last 100 episodes) against ε, one line per method.

In [ ]:
# TODO: average reward against epsilon, one line per method.
# your code here

**Questions**

1. How did SARSA's route change as ε grew? Did Q-learning's route change? Explain the difference using the two targets.

   *your answer here*

2. How did the gap in reward between the two methods change as ε grew? Why does Q-learning suffer more from exploration?

   *your answer here*

3. A robot will first be trained, then used with exploration switched off. Which method would you choose, and why? And if the robot must keep exploring while it works?

   *your answer here*

4. If any of your learned routes did not reach G, look at it with `plot_route`. What does the robot do instead?

   *your answer here*

---
## Part 4: Summary and Reflection

### Task 6: Summary and Reflection

**(a)** Table 9: The methods of Labs 5 to 9

| Method | Needs the rules? | Learns V or Q? | Prediction or control? | On-policy or off-policy? |
|---|---|---|---|---|
| DP policy evaluation (Lab 5) | | | | — |
| Monte Carlo prediction (Lab 7) | | | | — |
| TD(0) prediction (Lab 8) | | | | — |
| SARSA (Lab 9) | | | | |
| Q-learning (Lab 9) | | | | |

**(b)** In no more than 150 words, explain which method you would use to train a real robot working next to a dangerous dock edge, and why. Quote at least two numbers from your own results.

*your answer here (max 150 words)*